# 04 Vector Database

## Objective

This notebook stores embeddings in a vector database and performs semantic similarity search.

## Goals

- Generate embeddings
- Create a FAISS index
- Perform similarity search
- Retrieve relevant document chunks

## Use Case

Fraud Detection Knowledge Assistant

In [3]:
!pip install pypdf pandas sentence_transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 32.5 MB/s eta 0:00:00


In [5]:
import faiss
import pandas as pd

from pathlib import Path
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader

# PDF Text Extraction Function
def extract_pdf_text(pdf_path):
  reader = PdfReader(pdf_path)
  text = ""
  for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
      text += page_text + "\n"
  return text

# Text Chunking Function
def create_chunks(text, chunk_size=1000):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])
  return chunks

# Load Documents
RAW_DATA_DIR = Path("data/raw")
pdf_files = list(RAW_DATA_DIR.glob("*.pdf"))
documents = []
for pdf_file in pdf_files:
  text = extract_pdf_text(pdf_file)
  documents.append({"file_name": pdf_file.name, "text": text})
documents_df = pd.DataFrame(documents)

# Create Chunks
chunk_records = []
for _, row in documents_df.iterrows():
  chunks = create_chunks(row["text"])
  for idx, chunk in enumerate(chunks):
    chunk_records.append({"file_name": row["file_name"], "chunk_id": idx, "chunk_text": chunk})
chunks_df = pd.DataFrame(chunk_records)
print(f"Total chunks: {len(chunks_df)}")

# Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

# Generate Embeddings
embeddings = model.encode(chunks_df["chunk_text"].tolist(), show_progress_bar=True)

# Build FAISS Index
embedding_dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dimension)
index.add(embeddings)
print(f"Vectors stored: {index.ntotal}")

# Similarity Search
query = "How does the paper detect fraud?"
query_embedding = model.encode([query])
distances, indices = index.search(query_embedding, k=3)
print(indices)

# Display Results
for position, idx in enumerate(indices[0]):
  print("=" * 80)
  print(f"Results {position +1}")
  print(f"Distance: {distances[0][position]}")
  print()
  print(chunks_df.iloc[idx]["chunk_text"][:1000])
  print()

Total chunks: 88


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Vectors stored: 88
[[8 3 4]]
Results 1
Distance: 0.9880292415618896

interpretability of model decisions, thus enhancing trust and transparency.
 5. Error analysis: We perform a detailed error analysis to understand failure cases and provide directions for 
further model refinement.
Together, these contributions not only advance predictive performance but also improve transparency, efficiency, 
and trustworthiness of fraud detection systems.
The rest of the paper is organized as follows. Section II presents related work. Section III describes the 
proposed approach for detecting credit card fraud. The experiments conducted and their analyses are presented 
in Section IV . Section V provides the computational cost analysis. Section VI presents the model generalization 
assessment for the proposed ensemble models. Section VII discusses the explainable AI (XAI) analysis. Section 
VIII contains the error analysis and model comparison. Section IX outlines the research limitations. Finally, 

## Conclusion

Embeddings were successfully stored in a FAISS vector database.

Semantic similarity search can now retrieve relevant document chunks for user questions.

The next notebook will combine retrieval with a language model to build a complete RAG pipeline.